In [0]:
#%pip install xlsxwriter

In [0]:
%run ../config/utils

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import Row
from datetime import     datetime
from pyspark.sql.types import StructType, StructField, DateType, StringType, DoubleType


w = WorkspaceClient()  # uses cluster/job auth automatically

run_date = datetime.today().date()

In [0]:
#spark.sql(f"SELECT * FROM {pipeline_summary_report}").display()

In [0]:
# Get the max date from the pipeline_summary_report table 
#   we are filtering for dates that are less than today in case we run this process manually several times the same day

last_date = spark.sql(f"SELECT MAX(run_date) AS max_date FROM {pipeline_summary_report}  WHERE run_date < '{run_date}'").collect()[0]['max_date']

print(f'Last date registered: {last_date}')

# Retrieve latest records
if last_date:
    latest_records = spark.sql(f"SELECT * FROM {pipeline_summary_report} WHERE run_date = '{last_date}'")
    #display(latest_records)
else:
    schema = StructType([
        StructField("run_date", DateType(), True),
        StructField("job_name", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("duration_in_hs", DoubleType(), True),
        StructField("delta_to_prev_week", StringType(), True),
        StructField("URL_of_job_run", StringType(), True)
    ])

    latest_records = spark.createDataFrame([], schema)

In [0]:

schema = StructType([
    StructField("run_date", DateType(), True),
    StructField("job_name", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("duration_in_hs", DoubleType(), True),
    StructField("delta_to_prev_week", StringType(), True),
    StructField("URL_of_job_run", StringType(), True)
])

empty_df = spark.createDataFrame([], schema)

In [0]:
# List the job/pipelines to be monitored

job_names = ['de_wf_ds__Personalization__pe_engine_orchestration', # This one includes every step below
             'de_wf_ds__Personalization__etl',
             'de_wf_ds__Personalization__category_dna',
             'de_wf_ds__Personalization__member_dna',
             'de_wf_ds__Personalization__model__bbm',
             'de_wf_ds__Personalization__model__cf_model',
             'de_wf_ds__Personalization__model__gm_execution',
             'de_wf_ds__Personalization__model__trip_spend',
             'de_wf_ds__Personalization__model__digital_propensity'
             ]  # add any new pipeline you need to track here


rows = []


# We will query the info for each job/pipeline
for job_name in job_names:

    pipeline_runs_for_this_job_name = list(w.jobs.list(name=job_name))

    if not pipeline_runs_for_this_job_name:
        continue
    job_id = pipeline_runs_for_this_job_name[0].job_id # get the last one

    runs = list(w.jobs.list_runs(job_id=job_id, limit=1))

    if not runs:
        continue

    r = runs[0]

    rows.append(Row(
        run_date    = run_date,
        job_name    = r.run_name,
        start_time  = datetime.utcfromtimestamp(r.start_time / 1000).strftime("%Y-%m-%d %H:%M:%S"),
        end_time    = datetime.utcfromtimestamp(r.end_time   / 1000).strftime("%Y-%m-%d %H:%M:%S"),
        duration_in_hs   = round((r.run_duration)/ 1000 / 3600, 2),
        delta_to_prev_week = "None",
        URL_of_job_run     = r.run_page_url
    ))

curr_week_df_wo_perc = spark.createDataFrame(rows)

In [0]:
from pyspark.sql import functions as F

# --- Prep "prev" side with a clear column name ---
prev_df = (
    latest_records.select(
        F.col("job_name"),
        F.col("duration_in_hs").alias("prev_duration_in_hs")
    )
    .alias("prev")
)

merged_df = (
    curr_week_df_wo_perc.alias("curr")
    .join(prev_df, on="job_name", how="left")
)

# --- Handy refs ---
curr = F.col("curr.duration_in_hs")
prev = F.col("prev.prev_duration_in_hs")
pct  = (curr - prev) / prev  # only used when prev != 0

# --- Build message ---
delta_expr = (
    F.when(prev.isNull(), F.lit("None"))
     .when(prev == 0,
           F.when(curr == 0, F.lit("0% i.e time duration is same"))
            .otherwise(F.lit("new (prev=0)")))
     .otherwise(
         F.concat_ws(
             " ",
             F.format_string("%.2f%%", F.round(F.abs(pct) * 100, 2)),
             F.when(pct > 0,  F.lit("time increased"))
              .when(pct == 0, F.lit("i.e time duration is same"))
              .otherwise(     F.lit("time decreased"))
         )
     )
)

# --- Final DF: keep only current-week columns + the new message ---
curr_week_df = (
    merged_df
    .withColumn("delta_to_prev_week", delta_expr)
    .select("curr.*", "delta_to_prev_week")
)

display(curr_week_df)

In [0]:
curr_week_df.write.mode("overwrite").option('replaceWhere', f"run_date = '{run_date}'").saveAsTable(pipeline_summary_report)

In [0]:





# last_row = {
#     "job_name": "Total",
#     "build_num": "--",
#     "duration_in_min": t_min_current,
#     "duration_in_hours": t_hours_current,
#     "start_time_in_EST": "--",
#     "URL": "--",
#     "end_time_in_EST": "--",
#     "Gap_in_hours": g_t,
#     "delta_to_prev_week": delta,
# }

# merge_df = merge_df.append(last_row, ignore_index=True)

# merge_df.drop(["duration_in_min"], axis=1, inplace=True)


# # Arranging columns
# # job_name  start_time(EST) end_time(EST)   duration(h) rerun_gap(h)    delta_to_prev_week  build_num URL
# merge_df = merge_df[
#     [
#         "job_name",
#         "start_time_in_EST",
#         "end_time_in_EST",
#         "duration_in_hours",
#         "Gap_in_hours",
#         "delta_to_prev_week",
#         "build_num",
#         "URL",
#     ]
# ]


# merge_df.rename(
#     columns={
#         "duration_in_hours": "duration(h)",
#         "Gap_in_hours": "rerun_gap(h)",
#         "start_time_in_EST": "start_time(EST)",
#         "end_time_in_EST": "end_time(EST)",
#     },
#     inplace=True,
# )

# return merge_df


In [0]:
import xlsxwriter
from datetime import datetime

# File name
output_file_name = f'PE_dataload_summary_{run_date:%Y%m%d}.xlsx'
file_temp_path   = f'/Workspace/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/{output_file_name}'

# Create workbook/worksheet
workbook = xlsxwriter.Workbook(file_temp_path)
worksheet = workbook.add_worksheet("Summary")

# Define formats
header_format = workbook.add_format({"bold": True, "fg_color": "#D7E4BC", "border": 1})
increased_format = workbook.add_format({"fg_color": "#bfff80", "border": 1})
decreased_format = workbook.add_format({"fg_color": "#ffc561", "border": 1})

# Get schema and data
cols = curr_week_df.columns
data = curr_week_df.collect()  

# Write header
for col_num, col_name in enumerate(cols):
    worksheet.write(0, col_num, col_name, header_format)

# Write rows with conditional color
delta_idx = cols.index("delta_to_prev_week")

for i, row in enumerate(data, start=1):
    for j, col_name in enumerate(cols):
        val = row[col_name]
        fmt = None
        if j == delta_idx and isinstance(val, str):
            if "increased" in val:
                fmt = increased_format
            elif "decreased" in val:
                fmt = decreased_format
        worksheet.write(i, j, val, fmt)

workbook.close()

print(f"Excel file written to temp location: {file_temp_path}")

In [0]:
from shutil import move

# In DBX the excel lib cant be directly save to volumes so we store to a local tmp file and then move it to the volume with this code 

volume_path = f'/Volumes/{catalog_name}/{pe_schema_name}/job_summary_reports/{output_file_name}'  

move(file_temp_path, volume_path)

In [0]:
dbutils.jobs.taskValues.set(key="output_file_path", value=volume_path)

In [0]:
# # Using colour coding for merge_df
# # Create a Pandas Excel writer using XlsxWriter as the engine.
# file_name = "PE_dataload_summary_" + y + m + d + ".xlsx"

# writer = pd.ExcelWriter(file_name, engine="xlsxwriter")




# # Convert the dataframe to an XlsxWriter Excel object. Note that we turn off
# # the default header and skip one row to allow us to insert a user defined
# # header.

# merge_df.to_excel(
#     writer, sheet_name="Summary", startrow=0, header=True, index=False
# )

# # Get the xlsxwriter workbook and worksheet objects.
# workbook = writer.book
# worksheet = writer.sheets["Summary"]


# # Add a header format.
# header_format = workbook.add_format(
#     {
#         "bold": True,
#         "text_wrap": True,
#         "valign": "top",
#         "fg_color": "#D7E4BC",
#         "border": 1,
#     }
# )
# increased_format = workbook.add_format(
#     {"text_wrap": True, "valign": "top", "fg_color": "#bfff80", "border": 1}
# )
# decreased_format = workbook.add_format(
#     {"text_wrap": True, "valign": "top", "fg_color": "#ffc561", "border": 1}
# )


# # Bolding the headers.
# for col_num, value in enumerate(merge_df.columns.values):
#     worksheet.write(0, col_num, value, header_format)
# s = "delta_to_prev_week"  # column to be coloured
# l = list(merge_df.columns)
# ind = l.index("delta_to_prev_week")



# # Colouring the delta_to_prev_week column based on its value
# for i in range(0, len(merge_df)):
#     if merge_df["delta_to_prev_week"][i].find("increased") >= 0:
#         worksheet.write(
#             i + 1, ind, merge_df["delta_to_prev_week"][i], increased_format
#         )
#     else:
#         if merge_df["delta_to_prev_week"][i].find("decreased") >= 0:
#             worksheet.write(
#                 i + 1, ind, merge_df["delta_to_prev_week"][i], decreased_format
#             )
#         else:
#             worksheet.write(i + 1, ind, merge_df["delta_to_prev_week"][i])


# # Close the Pandas Excel writer and output the Excel file.
# writer.close()

